# Figure 3: Unseen-Control Benchmark

- Figure panels in the manuscript: `Fig3a` to `Fig3e`
- Notebook outputs: standalone main-text PNGs for unified all-model benchmark panels, one shared legend, and one representative single-perturbation case plot
- Inputs: `artifacts/results/<dataset>/metrics_*.csv`, `artifacts/results/<baseline>/<dataset>/metrics.csv`, payload `pkl`, raw dataset `h5ad`
- Outputs: `artifacts/paper_figures/main/Fig3_UnseenControlBenchmark/*`
- Run order: run after the core benchmark results for the selected single-perturbation datasets are ready
- Default datasets: `adamson`, `norman`, `dixit`
- Role: Main text benchmark only


In [ ]:
from __future__ import annotations

import importlib
import subprocess
import sys
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis
from scripts.trishift.analysis import baseline_panel

apply_gears_paper_style(font_scale=1.0)
from scripts.trishift.analysis._result_adapter import load_payload_item
from trishift._utils import load_adata, load_yaml, normalize_condition
importlib.reload(baseline_panel)


## Plan

- `Fig3a`: unified all-model `mean_pearson`
- `Fig3b`: unified all-model `mean_nmse`
- `Fig3c`: unified all-model `mean_systema_corr_20de_allpert`
- `Fig3e`: one representative single-perturbation case plot for qualitative support
- Shared legend is exported as a standalone figure for the benchmark bar charts
- Distribution-oriented panels are handled in `Fig5` or supplementary figures, not here
- Shared-baseline case uses a manually specified perturbation condition and the displayed 12-gene direction-aware expression profile (pred - control)


In [ ]:
DATASET_SPECS = [
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson', 'benchmark_subgroup_filter': None},
    {'dataset_key': 'norman', 'dataset_label': 'Norman', 'benchmark_subgroup_filter': None},
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit', 'benchmark_subgroup_filter': None},
]
SCGEN_DATASET_SPEC = {'dataset_key': 'scgen_pbmc_celltype', 'dataset_label': 'scGen'}
DATASET_ORDER = ['Adamson', 'Norman', 'Dixit', 'scGen']
BENCHMARK_MODELS = ['trishift_nearest', 'gears', 'biolord', 'genepert', 'scgpt', 'systema_nonctl_mean']
SCGEN_MODELS = ['trishift_nearest', 'biolord', 'scgpt']
MAIN_MODELS = ['trishift_nearest', 'gears', 'biolord', 'genepert', 'scgpt', 'systema_nonctl_mean']
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()
RESULT_MODE = 'unseen_ctrl'
SCGEN_RESULT_MODE = 'target_domain_ctrl'
PATHS_CFG = load_yaml(str(repo_root / 'configs' / 'paths.yaml'))

CASE_SHARED_BASELINE = {
    'dataset': 'adamson',
    'condition': 'CREB1+ctrl',
    'display_models': ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt'],
    'check_models': ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt'],
    'top_k': 12,
    'split_policy': 'first_available',
}
MODEL_LABELS = {
    'trishift_nearest': 'TriShift',
    'trishift_random': 'TriShift random',
    'biolord': 'biolord',
    'gears': 'GEARS',
    'genepert': 'GenePert',
    'scgpt': 'scGPT',
    'systema_nonctl_mean': 'Perturbed mean',
    'Truth': 'Truth',
}
MODEL_COLORS = {
    'TriShift': '#9FD9D3',
    'TriShift random': '#69B7B0',
    'biolord': '#F0806A',
    'GEARS': '#F2B56B',
    'GenePert': '#87A8D8',
    'scGPT': '#DDD3C8',
    'Perturbed mean': '#B9AEDC',
    'Truth': '#7F7F7F',
}
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig3_UnseenControlBenchmark'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
for pattern in ['fig3*.png', 'fig3*.csv']:
    for stale_path in OUT_ROOT.glob(pattern):
        stale_path.unlink()

print('Selected datasets:', [spec['dataset_key'] for spec in DATASET_SPECS] + [SCGEN_DATASET_SPEC['dataset_key']])
print('Selected splits:', SPLIT_IDS)
print('Result mode:', RESULT_MODE)
print('scGen result mode:', SCGEN_RESULT_MODE)
print('Shared-baseline case:', CASE_SHARED_BASELINE)

SCOUTER_STYLE_OMIT = {('dixit', 'biolord')}

def omit_scouter_style_rows(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    work = df.copy()
    dataset_col = 'dataset' if 'dataset' in work.columns else ('dataset_key' if 'dataset_key' in work.columns else None)
    model_col = 'model_name' if 'model_name' in work.columns else ('method' if 'method' in work.columns else None)
    if dataset_col is None or model_col is None:
        return work
    drop_mask = (
        work[dataset_col].astype(str).str.lower().eq('dixit')
        & work[model_col].astype(str).str.lower().eq('biolord')
    )
    return work.loc[~drop_mask].copy()


In [ ]:
def _read_csv_optional(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def safe_run_baseline(dataset_key: str, models: list[str], tag: str, subgroup_filter: str | None = None) -> dict[str, object]:
    out_dir = OUT_ROOT / f'{tag}_{dataset_key}'
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        '-m',
        'scripts.trishift.analysis.baseline_panel',
        '--dataset',
        dataset_key,
        '--models',
        ','.join(models),
        '--split_ids',
        ','.join(str(split_id) for split_id in SPLIT_IDS),
        '--out_root',
        str(out_dir),
        '--result_mode',
        RESULT_MODE,
    ]
    if subgroup_filter:
        cmd.extend(['--subgroup_filter', str(subgroup_filter)])
    try:
        completed = subprocess.run(cmd, cwd=repo_root, check=True, capture_output=True, text=True)
        result = {
            'summary_df': _read_csv_optional(out_dir / 'baseline_panel_summary.csv'),
            'raw_df': _read_csv_optional(out_dir / 'baseline_panel_raw.csv'),
            'subgroup_df': _read_csv_optional(out_dir / 'norman_subgroup_panel.csv'),
            'stdout': completed.stdout,
            'stderr': completed.stderr,
        }
        return {'status': 'ready', 'result': result, 'out_dir': out_dir}
    except Exception as exc:
        return {'status': 'pending', 'error': str(exc), 'out_dir': out_dir}


def _style_axis(ax: plt.Axes, *, grid_axis: str = 'y') -> None:
    paper_style_axis(ax, grid_axis=grid_axis)
    ax.set_axisbelow(True)


def _shrink_bars(ax: plt.Axes, scale: float = 0.9) -> None:
    for patch in ax.patches:
        width = patch.get_width()
        if width <= 0:
            continue
        new_width = width * scale
        patch.set_x(patch.get_x() + (width - new_width) / 2)
        patch.set_width(new_width)


def render_metric_group(summary_df: pd.DataFrame, metric: str, metric_label: str, models: list[str], out_path: Path, title: str) -> None:
    work = summary_df[summary_df['model_name'].isin(models)].copy()
    fig, ax = plt.subplots(figsize=(5.2, 3.9), dpi=240)
    if work.empty or f'mean_{metric}' not in work.columns:
        ax.text(0.5, 0.5, f'No rows for {metric}', ha='center', va='center')
        ax.axis('off')
    else:
        work['dataset_label'] = pd.Categorical(work['dataset_label'], DATASET_ORDER, ordered=True)
        work['plot_label'] = work['model_name'].map(MODEL_LABELS)
        work = work.sort_values(['dataset_label'])
        dataset_order = [label for label in DATASET_ORDER if label in set(work['dataset_label'].astype(str))]
        label_order = [MODEL_LABELS[m] for m in models if MODEL_LABELS[m] in set(work['plot_label'].astype(str))]
        palette = {label: MODEL_COLORS[label] for label in label_order}
        vals = work[f'mean_{metric}'].astype(float)
        ymin = float(vals.min())
        ymax = float(vals.max())
        span = max(ymax - ymin, max(abs(ymax), abs(ymin), 1e-6))
        lower = min(0.0, ymin - 0.08 * span)
        upper = ymax + 0.28 * span
        ax.set_ylim(lower, upper)
        seen_labels = []
        group_width = 0.72
        for dataset_idx, dataset_label in enumerate(dataset_order):
            sub = work[work['dataset_label'].astype(str) == str(dataset_label)].copy()
            present_labels = [label for label in label_order if label in set(sub['plot_label'].astype(str))]
            if not present_labels:
                continue
            base_width = min(0.13, group_width / max(len(present_labels), 1))
            bar_width = base_width * 0.82
            if len(present_labels) == 1:
                offsets = [0.0]
            else:
                offsets = np.linspace(-base_width * (len(present_labels) - 1) / 2, base_width * (len(present_labels) - 1) / 2, len(present_labels))
            for offset, label in zip(offsets, present_labels):
                row = sub[sub['plot_label'].astype(str) == str(label)]
                if row.empty:
                    continue
                value = float(row[f'mean_{metric}'].iloc[0])
                ax.bar(dataset_idx + float(offset), value, width=bar_width, color=palette[label], edgecolor='black', linewidth=0.5, label=label if label not in seen_labels else None)
                if label not in seen_labels:
                    seen_labels.append(label)
        ax.set_xticks(np.arange(len(dataset_order)))
        ax.set_xticklabels(dataset_order)
        ax.set_title(title, fontsize=10, pad=8)
        ax.set_xlabel('')
        ax.set_ylabel(metric_label)
        ax.tick_params(axis='x', rotation=32)
        _style_axis(ax)
        handles = [plt.Rectangle((0, 0), 1, 1, facecolor=palette[label], edgecolor='black', linewidth=0.5) for label in label_order]
        legend = ax.legend(handles, label_order, title='', frameon=False, ncol=3, loc='upper left', bbox_to_anchor=(0.02, 0.995), borderaxespad=0.0, handlelength=0.9, handletextpad=0.35, columnspacing=0.8)
        for txt in legend.get_texts():
            txt.set_fontsize(9)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)


def render_main_legend(out_path: Path) -> None:
    legend_order = ['TriShift', 'GEARS', 'biolord', 'GenePert', 'scGPT', 'Perturbed mean']
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=MODEL_COLORS[label], edgecolor='#444444', linewidth=0.7) for label in legend_order]
    fig, ax = plt.subplots(figsize=(8.6, 1.2), dpi=220)
    ax.axis('off')
    ax.legend(handles, legend_order, ncol=3, loc='center', frameon=False, fontsize=10, handlelength=1.0, handletextpad=0.4, columnspacing=1.4)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight', transparent=True)
    plt.close(fig)


def render_single_metric_heatmap(summary_df: pd.DataFrame, metric: str, title: str, cmap: str, out_path: Path) -> None:
    model_order = [MODEL_LABELS[m] for m in EXTRA_HEATMAP_MODELS]
    work = summary_df[summary_df['model_name'].isin(EXTRA_HEATMAP_MODELS)].copy()
    fig, ax = plt.subplots(figsize=(4.6, 4.8), dpi=220)
    col = f'mean_{metric}'
    if work.empty or col not in work.columns:
        ax.text(0.5, 0.5, f'No rows for {metric}', ha='center', va='center')
        ax.axis('off')
    else:
        work['dataset_label'] = pd.Categorical(work['dataset_label'], DATASET_ORDER, ordered=True)
        work['plot_label'] = work['model_name'].map(MODEL_LABELS)
        pivot = work.pivot_table(index='plot_label', columns='dataset_label', values=col, aggfunc='mean').reindex(index=model_order, columns=DATASET_ORDER)
        pivot = pivot.loc[pivot.notna().any(axis=1)]
        annot = pivot.copy().astype(object)
        for r in pivot.index:
            for c in pivot.columns:
                val = pivot.loc[r, c]
                annot.loc[r, c] = 'NA' if pd.isna(val) else f'{float(val):.3f}'
        ax.set_facecolor('#ECECEC')
        if pivot.shape[0] == 0 or not pivot.notna().to_numpy().any():
            ax.text(0.5, 0.5, 'No available values', ha='center', va='center')
            ax.axis('off')
        else:
            sns.heatmap(pivot, ax=ax, cmap=cmap, annot=annot, fmt='', linewidths=0.8, linecolor='white', cbar=True, mask=pivot.isna(), square=False)
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('')
            ax.set_ylabel('')
            ax.tick_params(axis='x', rotation=18)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)



def _mean_or_nan(df: pd.DataFrame, col: str) -> float:
    if col not in df.columns:
        return float('nan')
    values = pd.to_numeric(df[col], errors='coerce')
    return float(values.mean()) if values.notna().any() else float('nan')


def load_scgen_summary() -> pd.DataFrame:
    specs = [
        ('trishift_nearest', 'TriShift', repo_root / 'artifacts' / 'results' / 'scgen_pbmc_celltype' / 'metrics.csv'),
        ('biolord', 'biolord', repo_root / 'artifacts' / 'results' / 'biolord' / 'scgen_pbmc_celltype' / 'emb_scgen_ifnb1_zenodo_prott5' / 'metrics.csv'),
        ('scgpt', 'scGPT', repo_root / 'artifacts' / 'results' / 'scgpt' / 'scgen_pbmc_celltype' / 'metrics.csv'),
    ]
    rows = []
    for model_name, label, metrics_path in specs:
        if not metrics_path.exists():
            rows.append({
                'dataset': SCGEN_DATASET_SPEC['dataset_key'],
                'model_name': model_name,
                'label': label,
                'n_rows': 0,
                'split_ids_used': '',
                'metrics_path': str(metrics_path),
            })
            continue
        df = pd.read_csv(metrics_path)
        if 'split_id' in df.columns:
            df = df[pd.to_numeric(df['split_id'], errors='coerce').isin(SPLIT_IDS)].copy()
        row = {
            'dataset': SCGEN_DATASET_SPEC['dataset_key'],
            'model_name': model_name,
            'label': label,
            'n_rows': int(len(df)),
            'split_ids_used': ','.join(map(str, sorted(set(pd.to_numeric(df.get('split_id'), errors='coerce').dropna().astype(int).tolist())))) if 'split_id' in df.columns else '',
            'metrics_path': str(metrics_path),
        }
        for metric in ['pearson', 'nmse', 'systema_corr_20de_allpert', 'systema_corr_deg_r2']:
            row[f'mean_{metric}'] = _mean_or_nan(df, metric)
        rows.append(row)
    out = pd.DataFrame(rows)
    out['dataset_label'] = SCGEN_DATASET_SPEC['dataset_label']
    return out

@lru_cache(maxsize=None)
def _repo_path(path_like):
    path = Path(path_like)
    return path if path.is_absolute() else repo_root / path

def load_dataset_context(dataset_key: str):
    adata = load_adata(str(_repo_path(PATHS_CFG['datasets'][dataset_key])))
    cond_values = adata.obs['condition'].astype(str).map(normalize_condition).to_numpy()
    if 'gene_name' in adata.var.columns:
        gene_names = adata.var['gene_name'].astype(str).tolist()
    else:
        gene_names = adata.var_names.astype(str).tolist()
    gene_index = {gene_name: idx for idx, gene_name in enumerate(gene_names)}
    return adata, cond_values, gene_index


@lru_cache(maxsize=None)
def load_normalized_payload(dataset_key: str, model_name: str, split_id: int):
    _, payload = load_payload_item(dataset=dataset_key, model_name=model_name, split_id=int(split_id), condition=None, result_mode=RESULT_MODE)
    return {normalize_condition(str(k)): v for k, v in payload.items() if isinstance(v, dict)}


def resolve_case_item(dataset_key: str, condition_key: str, model_name: str, split_policy: str = 'first_available') -> dict[str, object]:
    if split_policy != 'first_available':
        raise ValueError(f'Unsupported split_policy: {split_policy}')
    errors: list[str] = []
    for split_id in SPLIT_IDS:
        try:
            payload = load_normalized_payload(dataset_key, model_name, int(split_id))
        except Exception as exc:
            errors.append(f'split {split_id}: {exc}')
            continue
        item = payload.get(condition_key)
        if item is not None:
            return {'model_name': model_name, 'status': 'ok', 'split_id': int(split_id), 'item': item, 'error': ''}
        errors.append(f'split {split_id}: condition missing')
    return {'model_name': model_name, 'status': 'missing', 'split_id': None, 'item': None, 'error': ' | '.join(errors[:4])}


def mean_vector(x) -> np.ndarray:
    return np.asarray(np.asarray(x, dtype=float).mean(axis=0)).reshape(-1)


def build_truth_reference(dataset_key: str, condition_key: str, reference_genes: list[str]) -> tuple[pd.Series, pd.Series]:
    adata, cond_values, gene_index = load_dataset_context(dataset_key)
    selected_genes = [gene_name for gene_name in reference_genes if gene_name in gene_index]
    if not selected_genes:
        raise ValueError(f'No reference genes could be aligned to raw dataset genes for {dataset_key}')
    idx = np.asarray([gene_index[gene_name] for gene_name in selected_genes], dtype=int)
    ctrl_mask = cond_values == 'ctrl'
    target_mask = cond_values == condition_key
    if int(target_mask.sum()) == 0:
        raise ValueError(f'Condition not found in raw data: {condition_key}')
    ctrl_mean = np.asarray(adata.X[ctrl_mask][:, idx].mean(axis=0)).reshape(-1).astype(float, copy=False)
    target_mean = np.asarray(adata.X[target_mask][:, idx].mean(axis=0)).reshape(-1).astype(float, copy=False)
    truth_delta = target_mean - ctrl_mean
    return pd.Series(truth_delta, index=selected_genes), pd.Series(ctrl_mean, index=selected_genes)


def extract_prediction_series(item: dict[str, object]) -> pd.Series:
    if item.get('Pred_full') is not None and item.get('gene_name_full') is not None:
        pred_mean = mean_vector(item['Pred_full'])
        pred_genes = np.asarray(item['gene_name_full']).astype(str)
    else:
        pred_mean = mean_vector(item['Pred'])
        pred_genes = np.asarray(item.get('DE_name', [f'g{i}' for i in range(pred_mean.shape[0])])).astype(str)
    return pd.Series(pred_mean, index=pred_genes).groupby(level=0).mean()


def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    x = np.asarray(a, dtype=float).reshape(-1)
    y = np.asarray(b, dtype=float).reshape(-1)
    if x.size == 0 or y.size == 0:
        return float('nan')
    if np.allclose(np.std(x), 0.0) or np.allclose(np.std(y), 0.0):
        return float('nan')
    return float(np.corrcoef(x, y)[0, 1])


def sign_agreement(truth: np.ndarray, pred: np.ndarray) -> float:
    truth_sign = np.sign(np.asarray(truth, dtype=float).reshape(-1))
    pred_sign = np.sign(np.asarray(pred, dtype=float).reshape(-1))
    if truth_sign.size == 0:
        return float('nan')
    return float(np.mean(truth_sign == pred_sign))


def _evaluate_case_condition(dataset_key: str, condition_key: str, display_models: list[str], check_models: list[str], split_policy: str, top_k: int):
    resolved = [resolve_case_item(dataset_key, condition_key, model_name, split_policy=split_policy) for model_name in check_models]
    resolved_df = pd.DataFrame([
        {
            'dataset': dataset_key,
            'condition': condition_key,
            'model_name': row['model_name'],
            'label': MODEL_LABELS.get(row['model_name'], row['model_name']),
            'status': row['status'],
            'split_id': row['split_id'],
            'error': row['error'],
        }
        for row in resolved
    ])
    resolved_ok = [row for row in resolved if row['status'] == 'ok' and row['model_name'] in display_models]
    if len(resolved_ok) < len(display_models):
        return None

    reference_item = resolved_ok[0]['item']
    reference_series = extract_prediction_series(reference_item)
    truth_series, ctrl_series = build_truth_reference(dataset_key, condition_key, reference_series.index.astype(str).tolist())

    pred_abs_series: dict[str, pd.Series] = {}
    common_gene_set = set(truth_series.index.astype(str))
    for row in resolved_ok:
        label = MODEL_LABELS.get(row['model_name'], row['model_name'])
        pred_series = extract_prediction_series(row['item'])
        pred_abs_series[label] = pred_series
        common_gene_set &= set(pred_series.index.astype(str))

    if len(common_gene_set) < top_k:
        return None

    common_gene_index = pd.Index(sorted(common_gene_set))
    truth_common = truth_series.reindex(common_gene_index).astype(float)
    ordered_genes = truth_common.abs().sort_values(ascending=False).index[:top_k].tolist()

    metrics_rows = []
    plot_rows = [pd.DataFrame({'Gene': ordered_genes, 'Expression': truth_series.reindex(ordered_genes).astype(float).to_numpy(), 'Group': 'Truth'})]
    for row in resolved_ok:
        label = MODEL_LABELS.get(row['model_name'], row['model_name'])
        pred_abs = pred_abs_series[label].reindex(ordered_genes).astype(float)
        pred_delta = pred_abs - ctrl_series.reindex(ordered_genes).astype(float)
        truth_sub = truth_series.reindex(ordered_genes).astype(float)
        diff = (pred_delta - truth_sub).abs()
        metrics_rows.append({
            'dataset': dataset_key,
            'condition': condition_key,
            'model_name': row['model_name'],
            'label': label,
            'split_id': row['split_id'],
            'pearson_to_truth': safe_corr(truth_sub.to_numpy(), pred_delta.to_numpy()),
            'mae_to_truth': float(diff.mean()) if len(diff) else float('nan'),
            'sign_agreement': sign_agreement(truth_sub.to_numpy(), pred_delta.to_numpy()),
        })
        plot_rows.append(pd.DataFrame({'Gene': ordered_genes, 'Expression': pred_delta.to_numpy(), 'Group': label}))

    plot_df = pd.concat(plot_rows, ignore_index=True)
    metrics_df = pd.DataFrame(metrics_rows).sort_values(['sign_agreement', 'pearson_to_truth', 'mae_to_truth'], ascending=[False, False, True]).reset_index(drop=True)

    tri_row = metrics_df.loc[metrics_df['model_name'] == 'trishift_nearest'].iloc[0]
    base_rows = metrics_df.loc[metrics_df['model_name'] != 'trishift_nearest']
    best_base_sign = float(base_rows['sign_agreement'].max())
    best_base_corr = float(base_rows['pearson_to_truth'].max())
    best_base_mae = float(base_rows['mae_to_truth'].min())
    meta = {
        'dataset': dataset_key,
        'condition': condition_key,
        'top_k': top_k,
        'ordered_genes': ordered_genes,
        'selected_splits': {row['model_name']: row['split_id'] for row in resolved},
        'tri_sign_gain': float(tri_row['sign_agreement']) - best_base_sign,
        'tri_corr_gain': float(tri_row['pearson_to_truth']) - best_base_corr,
        'tri_mae_gain': best_base_mae - float(tri_row['mae_to_truth']),
        'truth_abs_mean': float(truth_series.reindex(ordered_genes).abs().mean()),
    }
    return meta, plot_df, metrics_df, resolved_df


def build_case_bundle(case_cfg: dict[str, object]) -> tuple[dict[str, object], pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    dataset_key = str(case_cfg['dataset']).strip()
    display_models = list(case_cfg['display_models'])
    check_models = list(case_cfg.get('check_models', display_models))
    split_policy = str(case_cfg.get('split_policy', 'first_available')).strip()
    top_k = int(case_cfg.get('top_k', 12))
    condition_value = case_cfg.get('condition')
    if condition_value in (None, '', 'auto'):
        raise ValueError('CASE_SHARED_BASELINE.condition must be set explicitly')
    evaluated = _evaluate_case_condition(
        dataset_key,
        normalize_condition(str(condition_value).strip()),
        display_models,
        check_models,
        split_policy,
        top_k,
    )
    if evaluated is None:
        raise ValueError(f'Case condition could not be resolved: {condition_value}')
    meta, plot_df, metrics_df, resolved_df = evaluated
    return meta, plot_df, metrics_df, resolved_df


def render_case_plot(plot_df: pd.DataFrame, title: str, out_path: Path) -> None:
    plt.figure(figsize=(13.2, 5.2), dpi=220)
    ax = sns.barplot(data=plot_df, x='Gene', y='Expression', hue='Group', palette=MODEL_COLORS, errorbar=None, saturation=0.95)
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_linewidth(0.7)
    ax.set_xlabel('')
    ax.set_ylabel('Expression change over control')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=32)
    ax.legend(title='', frameon=False, ncol=min(3, plot_df['Group'].nunique()), loc='upper center', bbox_to_anchor=(0.5, 1.18))
    ax.axhline(y=0, color='#4A4A4A', linewidth=0.8, linestyle='-')
    _style_axis(ax)
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches='tight')
    plt.close()


In [ ]:
runs = []
for spec in DATASET_SPECS:
    benchmark_run = safe_run_baseline(spec['dataset_key'], BENCHMARK_MODELS, 'benchmark', subgroup_filter=spec.get('benchmark_subgroup_filter'))
    runs.append({**spec, 'benchmark_status': benchmark_run['status'], 'benchmark_run': benchmark_run})

availability_df = pd.DataFrame([{
    'dataset_key': row['dataset_key'],
    'dataset_label': row['dataset_label'],
    'benchmark_status': row['benchmark_status'],
    'benchmark_error': row['benchmark_run'].get('error', ''),
} for row in runs])
availability_df.to_csv(OUT_ROOT / 'dataset_availability.csv', index=False, encoding='utf-8-sig')
display(availability_df)

benchmark_frames = []
for row in runs:
    if row['benchmark_status'] == 'ready':
        df = row['benchmark_run']['result']['summary_df'].copy(); df['dataset_label'] = row['dataset_label']; benchmark_frames.append(df)
benchmark_summary_df = pd.concat(benchmark_frames, ignore_index=True) if benchmark_frames else pd.DataFrame()
benchmark_summary_df = omit_scouter_style_rows(benchmark_summary_df)
scgen_summary_df = load_scgen_summary()
benchmark_summary_df = pd.concat([benchmark_summary_df, scgen_summary_df], ignore_index=True) if not scgen_summary_df.empty else benchmark_summary_df
benchmark_summary_df.to_csv(OUT_ROOT / 'benchmark_summary_all.csv', index=False, encoding='utf-8-sig')
display(benchmark_summary_df.head())

render_metric_group(benchmark_summary_df, 'pearson', 'Pearson', MAIN_MODELS, OUT_ROOT / 'fig3a_mean_pearson_all_models.png', 'Cross-dataset benchmark | mean Pearson')
render_metric_group(benchmark_summary_df, 'nmse', 'nMSE', MAIN_MODELS, OUT_ROOT / 'fig3b_mean_nmse_all_models.png', 'Cross-dataset benchmark | mean nMSE')
render_metric_group(benchmark_summary_df, 'systema_corr_20de_allpert', 'Systema Pearson', MAIN_MODELS, OUT_ROOT / 'fig3c_mean_systema_corr_all_models.png', 'Cross-dataset benchmark | mean Systema Pearson')
render_main_legend(OUT_ROOT / 'fig3_legend_main.png')


In [ ]:
case_meta, case_plot_df, case_metrics_df, case_resolved_df = build_case_bundle(CASE_SHARED_BASELINE)
case_plot_df.to_csv(OUT_ROOT / 'fig3e_case_expression_values.csv', index=False, encoding='utf-8-sig')
case_metrics_df.to_csv(OUT_ROOT / 'fig3e_case_expression_metrics.csv', index=False, encoding='utf-8-sig')
case_resolved_df.to_csv(OUT_ROOT / 'fig3e_case_resolution.csv', index=False, encoding='utf-8-sig')
title = f"Fig3e: Shared-baseline case | {case_meta['dataset']} | {case_meta['condition']}"
render_case_plot(case_plot_df, title=title, out_path=OUT_ROOT / 'fig3e_case_expression.png')
print(case_meta)
display(case_metrics_df)
display(case_resolved_df)
display(case_plot_df.head(24))
print(OUT_ROOT)
